# MLOps & Model Deployment
Key MLOps concepts: versioning, CI/CD, monitoring, serving.
Prerequisites: `pip install numpy matplotlib scikit-learn pandas`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import json, hashlib, time, os
np.random.seed(42)

## 1. ML Lifecycle Overview
MLOps covers: Data → Train → Evaluate → Deploy → Monitor → Retrain

In [ ]:
stages = ['Data\nCollection', 'Feature\nEngineering', 'Model\nTraining',
          'Evaluation', 'Deployment', 'Monitoring', 'Retraining']
fig, ax = plt.subplots(figsize=(12, 3))
for i, stage in enumerate(stages):
    color = plt.cm.Set3(i / len(stages))
    ax.add_patch(plt.Rectangle((i*1.5, 0), 1.3, 1, facecolor=color, edgecolor='black'))
    ax.text(i*1.5 + 0.65, 0.5, stage, ha='center', va='center', fontsize=9, fontweight='bold')
    if i < len(stages) - 1:
        ax.annotate('', xy=((i+1)*1.5, 0.5), xytext=(i*1.5+1.3, 0.5),
                    arrowprops=dict(arrowstyle='->', color='black'))
# Loop arrow from Retraining back to Training
ax.annotate('', xy=(2*1.5, 1.1), xytext=(6*1.5+0.65, 1.1),
            arrowprops=dict(arrowstyle='->', color='red', lw=2, connectionstyle='arc3,rad=0.3'))
ax.set_xlim(-0.2, 10.5); ax.set_ylim(-0.3, 1.6); ax.axis('off')
ax.set_title('ML Lifecycle (with Retraining Loop)', fontsize=13); plt.tight_layout(); plt.show()

## 2. Experiment Tracking
Track hyperparameters, metrics, and artifacts for reproducibility.

In [ ]:
class ExperimentTracker:
    """Minimal experiment tracker (like MLflow/W&B)."""
    def __init__(self):
        self.runs = []
    
    def log_run(self, name, params, metrics, model_hash):
        run = {
            'name': name,
            'params': params,
            'metrics': metrics,
            'model_hash': model_hash,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        }
        self.runs.append(run)
        return run
    
    def best_run(self, metric='accuracy'):
        return max(self.runs, key=lambda r: r['metrics'].get(metric, 0))

# Generate synthetic data
X = np.random.randn(500, 5)
y = (X[:, 0] + X[:, 1] > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

tracker = ExperimentTracker()

for C in [0.01, 0.1, 1.0, 10.0]:
    model = LogisticRegression(C=C, max_iter=200)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    model_hash = hashlib.md5(str(model.coef_).encode()).hexdigest()[:8]
    tracker.log_run(f'logreg_C{C}', {'C': C}, {'accuracy': acc}, model_hash)
    print(f'C={C:>5} | Accuracy={acc:.4f} | Hash={model_hash}')

print(f"\nBest run: {tracker.best_run()['name']} "
      f"(acc={tracker.best_run()['metrics']['accuracy']:.4f})")

## 3. Model Versioning & Registry

In [ ]:
class ModelRegistry:
    """Simple model registry (like MLflow Model Registry)."""
    def __init__(self):
        self.models = {}  # name → list of versions
    
    def register(self, name, version, metrics, stage='staging'):
        self.models.setdefault(name, []).append({
            'version': version, 'metrics': metrics, 'stage': stage
        })
    
    def promote(self, name, version, stage='production'):
        for v in self.models.get(name, []):
            if v['version'] == version:
                v['stage'] = stage
    
    def get_production(self, name):
        for v in self.models.get(name, []):
            if v['stage'] == 'production':
                return v
        return None

registry = ModelRegistry()
registry.register('fraud_detector', 'v1', {'f1': 0.82})
registry.register('fraud_detector', 'v2', {'f1': 0.89})
registry.register('fraud_detector', 'v3', {'f1': 0.91})
registry.promote('fraud_detector', 'v3')

print('All versions:')
for v in registry.models['fraud_detector']:
    print(f"  {v['version']} | F1={v['metrics']['f1']} | Stage={v['stage']}")
print(f"\nProduction model: {registry.get_production('fraud_detector')}")

## 4. Data & Model Drift Detection

In [ ]:
# Simulate data drift: training distribution vs production distribution
np.random.seed(42)
train_dist = np.random.normal(loc=0, scale=1, size=1000)
prod_dist_ok = np.random.normal(loc=0.1, scale=1.05, size=1000)  # slight drift
prod_dist_bad = np.random.normal(loc=1.5, scale=2.0, size=1000)  # major drift

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, prod, title in zip(axes, [prod_dist_ok, prod_dist_bad],
                            ['Minor Drift (OK)', 'Major Drift (Alert!)']):
    ax.hist(train_dist, bins=40, alpha=0.5, label='Training', density=True)
    ax.hist(prod, bins=40, alpha=0.5, label='Production', density=True)
    ax.set_title(title); ax.legend()
plt.suptitle('Data Drift Detection', fontsize=14); plt.tight_layout(); plt.show()

# PSI (Population Stability Index) for drift quantification
def compute_psi(expected, actual, bins=10):
    breakpoints = np.linspace(min(expected.min(), actual.min()),
                              max(expected.max(), actual.max()), bins+1)
    exp_pct = np.histogram(expected, breakpoints)[0] / len(expected) + 1e-6
    act_pct = np.histogram(actual, breakpoints)[0] / len(actual) + 1e-6
    return np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))

print(f'PSI (minor drift): {compute_psi(train_dist, prod_dist_ok):.4f}  (< 0.1 = stable)')
print(f'PSI (major drift): {compute_psi(train_dist, prod_dist_bad):.4f}  (> 0.2 = significant drift)')

## 5. A/B Testing for Model Deployment

In [ ]:
def ab_test_simulation(n_requests=10000, model_a_acc=0.85, model_b_acc=0.88, traffic_split=0.5):
    """Simulate A/B testing between two models."""
    results = {'A': [], 'B': []}
    for _ in range(n_requests):
        if np.random.random() < traffic_split:
            results['A'].append(np.random.random() < model_a_acc)
        else:
            results['B'].append(np.random.random() < model_b_acc)
    
    acc_a = np.mean(results['A'])
    acc_b = np.mean(results['B'])
    return acc_a, acc_b, len(results['A']), len(results['B'])

acc_a, acc_b, n_a, n_b = ab_test_simulation()
print(f'Model A: accuracy={acc_a:.4f} (n={n_a})')
print(f'Model B: accuracy={acc_b:.4f} (n={n_b})')
print(f'Difference: {(acc_b - acc_a)*100:.2f}% in favor of B')

# Statistical significance
from scipy import stats
z_stat = (acc_b - acc_a) / np.sqrt(acc_a*(1-acc_a)/n_a + acc_b*(1-acc_b)/n_b)
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
print(f'Z-statistic: {z_stat:.2f}, p-value: {p_value:.4f}')
print(f'Result: {"Significant" if p_value < 0.05 else "Not significant"} at alpha=0.05')

## 6. Interview Takeaways
- **MLOps** = ML + DevOps: automate the full ML lifecycle
- **Experiment tracking** (MLflow, W&B): log params, metrics, artifacts
- **Model registry**: version models, promote staging → production
- **Data drift** (PSI, KS test): monitor input distribution changes
- **Model drift**: monitor prediction quality degradation
- **Deployment patterns**: shadow mode, canary, blue-green, A/B testing
- **CI/CD for ML**: automated training, validation, deployment pipelines

---

<small><em>© 2026 AI Nirvana · Disclaimer: Provided as is. No liability assumed.</em></small>